# Silver Layer — Order Items Data Cleaning & Quality Checks
**GlobalMart | Tredence DE Advanced Training**

| | |
|---|---|
| **Source** | `gbmart.bronze.order_items` (via Lakeflow Connect CDC from Supabase) |
| **Target** | `gbmart.silver.order_items` |
| **SCD Type** | SCD1 — line items are transactional facts, same as orders |

> **Note on referential integrity:** we initially found 352 `order_items`
> orphaned because `silver.orders` had quarantined 114 parent orders for
> a `DELIVERY_BEFORE_SHIP` timezone artifact. That decision was reversed
> in the orders notebook (flag instead of quarantine — see
> `02_orders_data_cleaning_dq_checks.ipynb` Step 5), which resolved this
> cascade at the source. The check below confirms 0 orphans remain.

### What this notebook does
| Step | Action |
|---|---|
| 1 | Setup |
| 2 | Read Bronze + inspect schema |
| 3 | DQ scan — nulls, invalid quantity |
| 4 | `orderitemid` uniqueness check |
| 5 | Referential integrity check against `silver.orders` / `silver.products` |
| 6 | Transform + write |
| 7 | Verify |

## Step 1 — Setup

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

CATALOG      = "harsh_kumar01_npmentorskool_onmicrosoft_com"
BRONZE_TABLE = "harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.order_items"
SILVER_TABLE = "harsh_kumar01_npmentorskool_onmicrosoft_com.silver.order_items"

print(f"Reading from : {BRONZE_TABLE}")
print(f"Writing to   : {SILVER_TABLE}")

Reading from : harsh_kumar01_npmentorskool_onmicrosoft_com.bronze.order_items
Writing to   : harsh_kumar01_npmentorskool_onmicrosoft_com.silver.order_items


## Step 2 — Read Raw Data from Bronze

In [0]:
bronze_df = spark.table(BRONZE_TABLE)
print(f"Total records in Bronze: {bronze_df.count():,}")
print(f"Columns: {bronze_df.columns}")
bronze_df.display()

Total records in Bronze: 126,036
Columns: ['OrderItemID', 'OrderID', 'ProductID', 'Quantity', 'updated_at']


OrderItemID,OrderID,ProductID,Quantity,updated_at
OR-CS-000009,OR-000003,PRD-00005,3,2026-07-10T03:45:31.126135Z
OR-MIPA-000010,OR-000004,PRD-00198,1,2026-07-10T03:45:31.126135Z
OR-Y-000032,OR-000010,PRD-00236,2,2026-07-10T03:45:31.126135Z
OR-TP-000036,OR-000011,PRD-00034,3,2026-07-10T03:45:31.126135Z
OR-EW-000053,OR-000018,PRD-00002,5,2026-07-10T03:45:31.126135Z
OR-S-000074,OR-000026,PRD-00001,4,2026-07-10T03:45:31.126135Z
OR-S-000094,OR-000032,PRD-00334,3,2026-07-10T03:45:31.126135Z
OR-C-000097,OR-000034,PRD-00025,3,2026-07-10T03:45:31.126135Z
OR-HCA-000100,OR-000035,PRD-00007,1,2026-07-10T03:45:31.126135Z
OR-SC-000104,OR-000036,PRD-00001,4,2026-07-10T03:45:31.126135Z


## Step 3 — DQ Scan

| Rule | What we check | Why |
|---|---|---|
| `NULL_ORDER_ITEM_ID` | `orderitemid` is null | Primary key |
| `NULL_ORDER_ID` | `orderid` is null | FK to `fact_orders`/`orders` |
| `NULL_PRODUCT_ID` | `productid` is null | FK to `dim_product` |
| `INVALID_QUANTITY` | `quantity` is null or <= 0 | Can't order zero/negative units |

In [0]:
dq_scan_df = bronze_df \
    .withColumn("_dq_issue",
        when(col("orderitemid").isNull(),                                  lit("NULL_ORDER_ITEM_ID"))
        .when(col("orderid").isNull(),                                    lit("NULL_ORDER_ID"))
        .when(col("productid").isNull(),                                  lit("NULL_PRODUCT_ID"))
        .when(col("quantity").isNull() | (col("quantity") <= 0),          lit("INVALID_QUANTITY"))
        .otherwise(lit(None))
    )

print("=== DQ Issues Found ===")
dq_scan_df.groupBy("_dq_issue").count().orderBy("count", ascending=False).display()

=== DQ Issues Found ===


_dq_issue,count
null,126036


**Finding: 0 DQ issues** across all 377,866 rows.

## Step 4 — Investigate: Is `orderitemid` Unique?

In [0]:
dupe_count = bronze_df.groupBy("orderitemid").count().filter("count > 1").count()
print(f"Duplicate orderitemid count: {dupe_count}")

Duplicate orderitemid count: 0


**Finding: 0 duplicates.**

## Step 5 — Referential Integrity Check
Unlike `orders` (where lookup tables didn't exist yet when we built that
notebook), `silver.orders` and `silver.products` already exist here. A
`left anti join` finds any `order_item` whose `order_id` or `product_id`
doesn't exist in its parent table.

In [0]:
orders_df   = spark.table("harsh_kumar01_npmentorskool_onmicrosoft_com.silver.orders")
products_df = spark.table("harsh_kumar01_npmentorskool_onmicrosoft_com.silver.products")

orphan_orders   = bronze_df.join(orders_df,   bronze_df.order_id   == orders_df.order_id,   "left_anti")
orphan_products = bronze_df.join(products_df, bronze_df.productid == products_df.product_id, "left_anti")

print(f"order_items with no matching order_id  : {orphan_orders.count():,} / {bronze_df.count():,}")
print(f"order_items with no matching product_id: {orphan_products.count():,} / {bronze_df.count():,}")

---------------------------------------------------------------------------
PySparkAttributeError                     Traceback (most recent call last)
File <command-6693285783294377>, line 4
      1 orders_df   = spark.table("harsh_kumar01_npmentorskool_onmicrosoft_com.silver.orders")
      2 products_df = spark.table("harsh_kumar01_npmentorskool_onmicrosoft_com.silver.products")
----> 4 orphan_orders   = bronze_df.join(orders_df,   bronze_df.order_id   == orders_df.order_id,   "left_anti")
      5 orphan_products = bronze_df.join(products_df, bronze_df.productid == products_df.product_id, "left_anti")
      7 print(f"order_items with no matching order_id  : {orphan_orders.count():,} / {bronze_df.count():,}")

File /databricks/spark/python/pyspark/databricks/instrumentation/instrumentation_utils.py:217, in _wrap_function.<locals>.wrapper(*args, **kwargs)
    215 start = time.perf_counter()
    216 try:
--> 217     res = func(*args, **kwargs)
    218     logging_helper.log_event(
    2

In [0]:
# Read Silver tables
orders_df = spark.table(
    "harsh_kumar01_npmentorskool_onmicrosoft_com.silver.orders"
)

products_df = spark.table(
    "harsh_kumar01_npmentorskool_onmicrosoft_com.silver.products"
)

# Find order_items whose OrderID doesn't exist in the Orders table
orphan_orders = bronze_df.join(
    orders_df,
    bronze_df["OrderID"] == orders_df["order_id"],
    "left_anti"
)

# Find order_items whose ProductID doesn't exist in the Products table
orphan_products = bronze_df.join(
    products_df,
    bronze_df["ProductID"] == products_df["product_id"],
    "left_anti"
)

# Print results
bronze_count = bronze_df.count()

print(f"Total Order Items                : {bronze_count:,}")
print(f"Missing OrderID references       : {orphan_orders.count():,}")
print(f"Missing ProductID references     : {orphan_products.count():,}")

Total Order Items                : 126,036
Missing OrderID references       : 0
Missing ProductID references     : 0


**Finding: 0 orphans on both checks.**

We previously found 352 orphans here, all tracing back to 114 orders that
`silver.orders` had quarantined for `DELIVERY_BEFORE_SHIP`. Quarantining
the *entire* parent order over a defect confined to two date columns
discarded real revenue/quantity data two tables downstream — so the orders
notebook was changed to flag those rows instead of removing them. That
fix resolved this cascade at the source; no quarantine logic is needed in
this notebook at all.

## Step 6 — Transform & Write
No derived columns needed — `order_items` is the grain of `fact_orders`;
pricing/totals get computed when joined with `products` at Gold.

In [0]:
silver_df = bronze_df \
    .withColumnRenamed("orderitemid", "order_item_id") \
    .withColumnRenamed("orderid", "order_id") \
    .withColumnRenamed("productid", "product_id") \
    .withColumn("_silver_updated_at", current_timestamp()) \
    .select("order_item_id", "order_id", "product_id", "quantity", "updated_at", "_silver_updated_at")



In [0]:
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(SILVER_TABLE)

print(f"Written to {SILVER_TABLE}: {spark.table(SILVER_TABLE).count():,} rows")

Written to harsh_kumar01_npmentorskool_onmicrosoft_com.silver.order_items: 126,036 rows


## Step 7 — Verify

In [0]:
df = spark.table(SILVER_TABLE)
print(f"order_items rows : {df.count():,}")  # should be 377,866 — full count, nothing lost

# Re-confirm 0 orphans against the final written table
final_orphan_check = df.join(
    spark.table("harsh_kumar01_npmentorskool_onmicrosoft_com.silver.orders").select("order_id"),
    df.order_id == col("order_id"), "left_anti"
)
print(f"Orphaned order_items in final silver table: {final_orphan_check.count():,}")  # should be 0

df.printSchema()

order_items rows : 126,036


{"ts": "2026-07-10 13:05:25.307", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[AMBIGUOUS_REFERENCE] Reference `order_id` is ambiguous, could be: [`harsh_kumar01_npmentorskool_onmicrosoft_com`.`silver`.`order_items`.`order_id`, `harsh_kumar01_npmentorskool_onmicrosoft_com`.`silver`.`orders`.`order_id`]. SQLSTATE: 42704", "context": {"file": "<command-6693285783294383>, line 7 in cell [24]", "line": "", "fragment": "col", "errorClass": "AMBIGUOUS_REFERENCE"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o765.join.\n: org.apache.spark.sql.AnalysisException: [AMBIGUOUS_REFERENCE] Reference `order_id` is ambiguous, could be: [`harsh_kumar01_npmentorskool_onmicrosoft_com`.`silver`.`order_items`.`order_id`, `harsh_kumar01_npmentorskool_onmicrosoft_com`.`silver`.`orders`.`order_id`]. SQLSTATE: 42704\n\tat org.apache.spark.sql.errors.QueryCompilationErrors$.ambiguousReferenceError(QueryCompilationErrors.scala:2995)\n\tat org.apache.spark

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6693285783294383>, line 5
      2 print(f"order_items rows : {df.count():,}")  # should be 377,866 — full count, nothing lost
      4 # Re-confirm 0 orphans against the final written table
----> 5 final_orphan_check = df.join(
      6     spark.table("harsh_kumar01_npmentorskool_onmicrosoft_com.silver.orders").select("order_id"),
      7     df.order_id == col("order_id"), "left_anti"
      8 )
      9 print(f"Orphaned order_items in final silver table: {final_orphan_check.count():,}")  # should be 0
     11 df.printSchema()

File /databricks/spark/python/pyspark/databricks/instrumentation/instrumentation_utils.py:217, in _wrap_function.<locals>.wrapper(*args, **kwargs)
    215 start = time.perf_counter()
    216 try:
--> 217     res = func(*args, **kwargs)
    218     logging_helper.log_event(
    219         accessor=wra

In [0]:
from pyspark.sql.functions import col

df = spark.table(SILVER_TABLE)

print(f"order_items rows : {df.count():,}")

orders_df = (
    spark.table("harsh_kumar01_npmentorskool_onmicrosoft_com.silver.orders")
    .select("order_id")
)

# Re-confirm there are no orphan order_items
final_orphan_check = (
    df.alias("oi")
      .join(
          orders_df.alias("o"),
          (col("oi.order_id") == col("o.order_id")) &
          col("oi.order_id").isNotNull(),
          "left_anti"
      )
)

print(f"Orphaned order_items in final silver table: {final_orphan_check.count():,}")

df.printSchema()

order_items rows : 126,036
Orphaned order_items in final silver table: 0
root
 |-- order_item_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- updated_at: timestamp (nullable = true)
 |-- _silver_updated_at: timestamp (nullable = true)



## Reset (if needed)

In [0]:
# spark.sql(f"DROP TABLE IF EXISTS {SILVER_TABLE}")
# print("Reset complete")